# Laboratorio 9 – Redes Neuronales Artificiales (RNA)

## Configuración – Reproducción del Pipeline de Labs 4–8

In [43]:
import pyreadr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     GridSearchCV, learning_curve)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, f1_score, precision_score,
                             recall_score, mean_squared_error, mean_absolute_error,
                             r2_score)
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.linear_model import LogisticRegression, RidgeCV
from sklearn.svm import SVC, SVR, LinearSVC

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110
print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


In [44]:
# Pipeline idéntico a Labs 4–8
result  = pyreadr.read_r('listings.Rdata')
df_raw  = result[list(result.keys())[0]].copy()
df      = df_raw.copy()

# Precio
if df['price'].dtype == object:
    df['price'] = (df['price'].str.replace(r'[\$,]', '', regex=True)
                               .str.strip().replace('', np.nan).astype(float))
q_high = df['price'].quantile(0.99)
df = df[(df['price'] > 0) & (df['price'] <= q_high)].copy()

# Eliminar columnas innecesarias
cols_drop = ['id','listing_url','scrape_id','last_scraped','source','name',
    'description','neighborhood_overview','picture_url','host_url',
    'host_thumbnail_url','host_picture_url','host_about','host_verifications',
    'amenities','calendar_updated','calendar_last_scraped','license',
    'bathrooms_text','minimum_minimum_nights','maximum_minimum_nights',
    'minimum_maximum_nights','maximum_maximum_nights',
    'minimum_nights_avg_ntm','maximum_nights_avg_ntm']
df = df.drop(columns=[c for c in cols_drop if c in df.columns])

# Eliminar columnas con >60% nulos
null_pct = df.isnull().mean()
df = df.drop(columns=null_pct[null_pct > 0.60].index.tolist())

# Feature engineering de fechas
if 'host_since' in df.columns:
    df['host_since'] = pd.to_datetime(df['host_since'], errors='coerce')
    df['host_years'] = ((pd.Timestamp('2024-01-01') - df['host_since']).dt.days / 365).round(1)
    df = df.drop(columns=['host_since'])
df = df.drop(columns=[c for c in ['first_review','last_review'] if c in df.columns], errors='ignore')

# Booleans → 0/1
for col in ['host_is_superhost','host_has_profile_pic','host_identity_verified',
            'has_availability','instant_bookable']:
    if col in df.columns:
        df[col] = df[col].map({'t':1,'f':0,True:1,False:0})

# Porcentajes
for col in ['host_response_rate','host_acceptance_rate']:
    if col in df.columns:
        df[col] = df[col].str.replace('%','',regex=False).str.strip()
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Encoding de variables categóricas
TARGET = 'price'
num_features = [c for c in df.select_dtypes(include='number').columns if c != TARGET]
cat_features  = [c for c in ['room_type','property_type','neighbourhood_cleansed',
                               'host_response_time'] if c in df.columns]
for col in cat_features:
    freq = df[col].value_counts(normalize=True)
    df[col] = df[col].replace(freq[freq < 0.01].index, 'Otro')

df_encoded = pd.get_dummies(df[num_features + cat_features + [TARGET]],
                             columns=cat_features, drop_first=True, dtype=int)
for col in df_encoded.columns:
    if df_encoded[col].isnull().any():
        df_encoded[col].fillna(df_encoded[col].median(), inplace=True)

# Variable respuesta categórica
p33 = df_encoded[TARGET].quantile(0.33)
p67 = df_encoded[TARGET].quantile(0.67)
df_encoded['price_category'] = df_encoded[TARGET].apply(
    lambda p: 'Económico' if p <= p33 else ('Intermedio' if p <= p67 else 'Caro'))

feature_cols = [c for c in df_encoded.columns if c not in [TARGET, 'price_category']]
X        = df_encoded[feature_cols]
y_price  = df_encoded[TARGET]

# Splits IDÉNTICOS a todos los labs anteriores
X_train, X_test, y_train_price, y_test_price = train_test_split(
    X, y_price, test_size=0.20, random_state=42)

le = LabelEncoder()
y_cat = le.fit_transform(df_encoded['price_category'])
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_cat, test_size=0.20, random_state=42, stratify=y_cat)

print(f"Dataset final: {df_encoded.shape[0]:,} filas x {len(feature_cols)} features")
print(f"Train completo: {len(X_train):,}  |  Test completo: {len(X_test):,}")
print(f"Umbral P33 = ${p33:.0f}  |  Umbral P67 = ${p67:.0f}")
print(f"Clases: {dict(zip(le.classes_, le.transform(le.classes_)))}")
print("Pipeline reproducido — idéntico a Labs 4-8.")

Dataset final: 75,531 filas x 75 features
Train completo: 60,424  |  Test completo: 15,107
Umbral P33 = $140  |  Umbral P67 = $267
Clases: {'Caro': 0, 'Económico': 1, 'Intermedio': 2}
Pipeline reproducido — idéntico a Labs 4-8.


## Normalización de Datos — Requisito Crítico para Redes Neuronales

A diferencia de Random Forest o Naive Bayes, las redes neuronales son **extremadamente sensibles
a la escala de los datos**. La razón es matemática:

Durante la **propagación hacia adelante (forward propagation)**, cada neurona calcula:

$$z = w_1 x_1 + w_2 x_2 + \ldots + w_n x_n + b$$

Si `x_1 = availability_365 ∈ [0, 365]` y `x_2 = review_score ∈ [1, 5]`, los pesos
`w_1` deben ser ~73x más pequeños que `w_2` para que la contribución sea equivalente.
**El gradiente descendente tiene dificultades para encontrar esos pesos tan asimétricos.**

El `StandardScaler` ya fue ajustado en Labs anteriores. Se reutiliza exactamente:

In [45]:
# StandardScaler — fit SOLO en train, transform en test
# Reutilizado idéntico a Lab 8 para garantizar comparabilidad total
scaler      = StandardScaler()
X_train_sc  = scaler.fit_transform(X_train)
X_test_sc   = scaler.transform(X_test)

# Los mismos datos escalados aplican para clasificación y regresión
# (RNA usa X_train_sc / X_test_sc en todos los experimentos)
print(f"X_train_sc shape : {X_train_sc.shape}")
print(f"X_test_sc  shape : {X_test_sc.shape}")
print(f"Media post-scaling (train) : {X_train_sc.mean(axis=0)[:5].round(4)}")
print(f"Std  post-scaling (train)  : {X_train_sc.std(axis=0)[:5].round(4)}")
print()
print("IMPORTANTE: Los datos de prueba se escalan con los parámetros del train set.")
print("Esto evita data leakage y garantiza una comparación válida con Labs anteriores.")

X_train_sc shape : (60424, 75)
X_test_sc  shape : (15107, 75)
Media post-scaling (train) : [-0.  0.  0. -0.  0.]
Std  post-scaling (train)  : [1. 1. 1. 1. 1.]

IMPORTANTE: Los datos de prueba se escalan con los parámetros del train set.
Esto evita data leakage y garantiza una comparación válida con Labs anteriores.


## Marco Teórico — Redes Neuronales Artificiales

### Arquitectura General

Una RNA está compuesta por capas de neuronas interconectadas:

```
Capa de entrada  →  Capas ocultas  →  Capa de salida
 [x1, x2, ..., xn]   [h1, h2, ...]     [ŷ_Económico, ŷ_Intermedio, ŷ_Caro]
```

### Forward Propagation

Para cada capa `l`:
$$a^{[l]} = g^{[l]}(W^{[l]} \cdot a^{[l-1]} + b^{[l]})$$

Donde `g` es la función de activación.

### Backpropagation

Minimiza la pérdida mediante gradiente descendente:
$$W := W - \alpha \cdot \frac{\partial \mathcal{L}}{\partial W}$$

### Funciones de Activación

| Función | Fórmula | Uso recomendado |
|---------|---------|-----------------|
| **ReLU** | `max(0, z)` | Capas ocultas — evita gradiente evanescente |
| **Tanh** | `(e^z - e^{-z})/(e^z + e^{-z})` | Capas ocultas — salida en [-1, 1] |
| **Sigmoid** | `1/(1+e^{-z})` | Clasificación binaria |
| **Softmax** | `e^{z_i}/Σe^{z_j}` | Capa de salida multiclase |

### Loss Functions

- **Cross-Entropy** (clasificación): $\mathcal{L} = -\sum y_i \log(\hat{y}_i)$
- **MSE** (regresión): $\mathcal{L} = \frac{1}{n}\sum(y_i - \hat{y}_i)^2$

## Modelos de Clasificación con RNA

### Diseño de Topologías

Para el problema SmartStay (clasificación en 3 categorías de precio), se diseñan
dos arquitecturas con filosofías distintas, entrenadas sobre 60,424 observaciones
con 75 features (umbral P33=$140, P67=$267):

**Modelo RNA-C1 — Arquitectura Profunda con ReLU**
- Topología: `[75 → 128 → 64 → 32 → 3]` (3 capas ocultas, ~24K parámetros)
- Activación ocultas: **ReLU** — evita el problema del gradiente evanescente, computacionalmente eficiente
- Activación salida: **Softmax** (implícita en `MLPClassifier`)
- Regularización L2: `alpha=0.001`
- Optimizador: **Adam** con `learning_rate_init=0.001`, `batch_size=256`
- Justificación: arquitectura en embudo que reduce progresivamente la dimensionalidad,
  extrayendo representaciones cada vez más abstractas del espacio de precios.

**Modelo RNA-C2 — Arquitectura Ancha con Tanh**
- Topología: `[75 → 200 → 100 → 3]` (2 capas ocultas, ~36K parámetros)
- Activación ocultas: **Tanh** — salida en [-1,1], simétrica alrededor del cero.
  Con datos escalados (media≈0), Tanh codifica relaciones negativas de forma más natural:
  por ejemplo, "a mayor `minimum_nights`, menor precio" se codifica directamente con pesos negativos.
- Regularización L2: `alpha=0.0001` (menor, para compensar la mayor capacidad)
- Justificación: capas más anchas capturan más interacciones de features en cada nivel.

Ambos modelos utilizan `early_stopping=True` con `n_iter_no_change=15` para
prevenir el sobreajuste, reservando el 10% del training set como validación interna.

In [46]:
# RNA-C1: Arquitectura Profunda con ReLU
# Topología: 128 → 64 → 32 | activación: relu | solver: adam
# adam (Adaptive Moment Estimation) adapta el learning rate por parámetro: velocidad de convergencia mayor que SGD en datos heterogéneos como Airbnb.
# early_stopping=True: detiene entrenamiento cuando val_loss no mejora → anti-overfitting.
# validation_fraction=0.1: 10% del train para monitoreo sin tocar el test.
# n_iter_no_change=15: paciencia antes de detener.

t0 = time.time()
rna_c1 = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu',
    solver='adam',
    alpha=0.001,
    batch_size=256,
    learning_rate_init=0.001,
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=15,
    random_state=42,
    verbose=False
)
rna_c1.fit(X_train_sc, y_train_c)
t_c1 = time.time() - t0

yp_c1_tr = rna_c1.predict(X_train_sc)
yp_c1_te = rna_c1.predict(X_test_sc)
acc_c1_tr = accuracy_score(y_train_c, yp_c1_tr)
acc_c1_te = accuracy_score(y_test_c,  yp_c1_te)
f1_c1_te  = f1_score(y_test_c, yp_c1_te, average='macro')

print("  RNA-C1 │ Arquitectura: 128→64→32 │ Activación: ReLU")
print(f"  Épocas efectivas     : {rna_c1.n_iter_}")
print(f"  Tiempo entrenamiento : {t_c1:.2f}s")
print(f"  Accuracy TRAIN       : {acc_c1_tr:.4f}")
print(f"  Accuracy TEST        : {acc_c1_te:.4f}")
print(f"  Gap (train-test)     : {acc_c1_tr - acc_c1_te:.4f}")
print(f"  F1-macro TEST        : {f1_c1_te:.4f}")
print()
print(classification_report(y_test_c, yp_c1_te, target_names=le.classes_, digits=4))

  RNA-C1 │ Arquitectura: 128→64→32 │ Activación: ReLU
  Épocas efectivas     : 19
  Tiempo entrenamiento : 7.05s
  Accuracy TRAIN       : 0.3789
  Accuracy TEST        : 0.3300
  Gap (train-test)     : 0.0488
  F1-macro TEST        : 0.3105

              precision    recall  f1-score   support

        Caro     0.3184    0.3414    0.3295      4962
   Económico     0.3207    0.1412    0.1960      4987
  Intermedio     0.3409    0.5017    0.4060      5158

    accuracy                         0.3300     15107
   macro avg     0.3267    0.3281    0.3105     15107
weighted avg     0.3269    0.3300    0.3116     15107



In [47]:
# RNA-C2: Arquitectura Ancha con Tanh
# Topología: 200 → 100 | activación: tanh
# Tanh es simétrica alrededor del cero (a diferencia de ReLU que es asimétrica).
# Con datos escalados (media≈0), tanh puede modela relaciones negativas de forma
# alpha menor (0.0001) → menos regularización para compensar la mayor capacidad de la red.

t0 = time.time()
rna_c2 = MLPClassifier(
    hidden_layer_sizes=(200, 100),
    activation='tanh',
    solver='adam',
    alpha=0.0001,
    batch_size=256,
    learning_rate_init=0.001,
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=15,
    random_state=42,
    verbose=False
)
rna_c2.fit(X_train_sc, y_train_c)
t_c2 = time.time() - t0

yp_c2_tr = rna_c2.predict(X_train_sc)
yp_c2_te = rna_c2.predict(X_test_sc)
acc_c2_tr = accuracy_score(y_train_c, yp_c2_tr)
acc_c2_te = accuracy_score(y_test_c,  yp_c2_te)
f1_c2_te  = f1_score(y_test_c, yp_c2_te, average='macro')

print("  RNA-C2 │ Arquitectura: 200→100 │ Activación: Tanh")
print(f"  Épocas efectivas     : {rna_c2.n_iter_}")
print(f"  Tiempo entrenamiento : {t_c2:.2f}s")
print(f"  Accuracy TRAIN       : {acc_c2_tr:.4f}")
print(f"  Accuracy TEST        : {acc_c2_te:.4f}")
print(f"  Gap (train-test)     : {acc_c2_tr - acc_c2_te:.4f}")
print(f"  F1-macro TEST        : {f1_c2_te:.4f}")
print()
print(classification_report(y_test_c, yp_c2_te, target_names=le.classes_, digits=4))

  RNA-C2 │ Arquitectura: 200→100 │ Activación: Tanh
  Épocas efectivas     : 18
  Tiempo entrenamiento : 20.32s
  Accuracy TRAIN       : 0.3580
  Accuracy TEST        : 0.3310
  Gap (train-test)     : 0.0270
  F1-macro TEST        : 0.3296

              precision    recall  f1-score   support

        Caro     0.3209    0.3325    0.3266      4962
   Económico     0.3291    0.2823    0.3039      4987
  Intermedio     0.3415    0.3765    0.3581      5158

    accuracy                         0.3310     15107
   macro avg     0.3305    0.3305    0.3296     15107
weighted avg     0.3306    0.3310    0.3299     15107

